In [12]:
%load_ext autoreload
%autoreload 2
from hyperparams import *
from tasks import *
from plot import *
from model import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# download_data()

In [4]:
mitdb, pwave = get_records(mitdb_dir, pwave_dir)

In [5]:
targets = []

In [6]:
# 전체 데이터를 저장할 리스트 초기화
all_segments = []
all_features = []
all_labels = []

# record별 전처리
for record in mitdb:
# for record in targets:
    # load ECG signal & annotations
    signals, fields = load_ECG_signal(record)
    signals = np.squeeze(signals)
    annotations = load_ECG_annotations(record=record, dir=mitdb_dir, extension='atr')
    
    # bandpass & R-peak detection with wavelet
    bpsig = bandpass_filter(signals)
    rpeaks = get_rpeaks(bpsig) 

    # segmentation
    segments = segment_heartbeats(bpsig, rpeaks) 

    # fill Nan, IQR clipping, remove constant features
    segments = fill_nan(segments)
    segments = IQR_clipping(segments) 
    segments = remove_const_features(segments)

    # P-peak detection
    ppeaks = get_ppeaks(record, bpsig, rpeaks)
    
    # remove 1st and last R-peak & match P,R peaks
    rpeaks = rpeaks[1:-1]
    ppeaks = match_pr(rpeaks, ppeaks)

    # feature extraction for 2nd input
    extracted_features = extract_features(bpsig, ppeaks, rpeaks)


    # normalization
    extracted_features = feature_scaling(extracted_features)
    segments = feature_scaling(segments)

    # label extraction & grouping
    labels = extract_labels(rpeaks, annotations, record)
    labels = list(map(group_labels, labels))

    # 데이터를 리스트에 추가
    all_segments.append(segments)
    all_features.append(extracted_features)
    all_labels.append(labels)
    
    
x1 = np.concatenate(all_segments, axis=0)
x2 = np.concatenate(all_features, axis=0)
y = np.concatenate(all_labels, axis=0)

print("Segments(x1) Shape:", x1.shape)
print("Extracted Features(x2) Shape:", x2.shape)
print("Labels(y) Shape:", y.shape)


rpeak: 649806, ann_idx: 1824 ann_sample length:1824, ann_symbol length:1824
rpeak: 649740, ann_idx: 2312 ann_sample length:2312, ann_symbol length:2312
rpeak: 649792, ann_idx: 2094 ann_sample length:2094, ann_symbol length:2094
rpeak: 649748, ann_idx: 2301 ann_sample length:2301, ann_symbol length:2301
rpeak: 649795, ann_idx: 2098 ann_sample length:2098, ann_symbol length:2098
rpeak: 649727, ann_idx: 2133 ann_sample length:2133, ann_symbol length:2133
rpeak: 649792, ann_idx: 1890 ann_sample length:1890, ann_symbol length:1890
Segments(x1) Shape: (129386, 300)
Extracted Features(x2) Shape: (129386, 16)
Labels(y) Shape: (129386,)


In [7]:
# data split
x1_train, x1_test_val, x2_train, x2_test_val, y_train, y_test_val = train_test_split(x1, x2, y, test_size=0.3, random_state=seed, stratify=y)
x1_val, x1_test, x2_val, x2_test, y_val, y_test = train_test_split(x1_test_val, x2_test_val, y_test_val, test_size=0.5, random_state=seed, stratify=y_test_val)


In [8]:
# one-hot encoding
y_train_oh = one_hot_encoder(y_train)
y_val_oh = one_hot_encoder(y_val)
y_test_oh = one_hot_encoder(y_test)

In [13]:
# model initialization
x1_shape = (x1_train.shape[1], 1)
x2_shape = (x2_train.shape[1],)
n_classes = y_train_oh.shape[1]
cnnx1only = CNNx1OnlyModel(x1_shape,n_classes)   

In [14]:
# model training
cnnx1only.fit(x1_train, y_train_oh, x1_val, y_val_oh, y_train)

Epoch 1/50
 110/2831 ━━━━━━━━━━━━━━━━━━━━ 1:45 39ms/step - accuracy: 0.3540 - loss: 5.2326

KeyboardInterrupt: 

In [ ]:
# model evaluation
test_loss, test_accuracy = cnnx1only.evaluate(x1_test, y_test_oh)
print(f"Test accuracy: {test_accuracy:.4f}")